# Step 2 — Structural Penalty Φ_S (Calculations S1–S3)

**Purpose:** Quantify structural constraints acting on each V-gene position:
- **S1** — Position-specific selection pressure ω_i: empirical dN/dS per Aho position using S5F as the neutral expectation
- **S2** — Forbidden mutations: positions where SHM generates mutations that selection removes (high expected, low observed frequency)
- **S3** — VH/VL interface co-variation: mutual information between Vernier zone positions across all paired memory sequences

**Input:** `processed/aligned_master.parquet`, `processed/mutability_profiles.parquet`  
**Outputs:** `results/tables/omega_per_position.parquet`, `results/tables/forbidden_mutations.parquet`, `results/tables/vh_vl_mutual_information.csv`, `results/figures/`  
**Design reference:** DESIGN.md §Step 2 (S1–S3)

In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
from collections import defaultdict
import time

In [ ]:
DATA_DIR  = Path("/home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/")
PROC_DIR  = DATA_DIR / "processed"
RESULTS   = DATA_DIR / "results"
FIGURES   = RESULTS / "figures"
TABLES    = RESULTS / "tables"

MASTER_FILE      = PROC_DIR / "aligned_master.parquet"
MUTABILITY_FILE  = PROC_DIR / "mutability_profiles.parquet"
MUTATED_FILE     = PROC_DIR / "iggnition_sequences.parquet"
GERMLINE_FILE    = PROC_DIR / "iggnition_germlines.parquet"

OMEGA_FILE   = TABLES / "omega_per_position.parquet"
FORBID_FILE  = TABLES / "forbidden_mutations.parquet"
MI_FILE      = TABLES / "vh_vl_mutual_information.csv"

print("Paths OK")

In [ ]:
master = pl.read_parquet(MASTER_FILE)
profile_H = pl.read_parquet(MUTABILITY_FILE)

memory = master.filter(~pl.col('naive_bio') & ~pl.col('naive_comp'))
KEY = "seq_name"
GAP = 45
INT_TO_NT = {65: 'A', 67: 'C', 71: 'G', 84: 'T'}

print(f"Master: {master.shape}")
print(f"Memory: {memory.height:,}")
print(f"Mutability profile: {profile_H.shape}")

In [ ]:
def aho_to_nt_cols(chain: str, aho_start: int, aho_end: int) -> list[str]:
    return [f"{chain}{i}" for i in range((aho_start - 1) * 3 + 1, aho_end * 3 + 1)]

H_REGIONS = {
    'FR1' : aho_to_nt_cols('H',   1,  25),
    'CDR1': aho_to_nt_cols('H',  26,  38),
    'FR2' : aho_to_nt_cols('H',  39,  49),
    'CDR2': aho_to_nt_cols('H',  50,  64),
    'FR3' : aho_to_nt_cols('H',  65, 108),
    'CDR3': aho_to_nt_cols('H', 109, 137),
    'FR4' : aho_to_nt_cols('H', 138, 149),
}
L_REGIONS = {
    'FR1' : aho_to_nt_cols('L',   1,  25),
    'CDR1': aho_to_nt_cols('L',  26,  38),
    'FR2' : aho_to_nt_cols('L',  39,  49),
    'CDR2': aho_to_nt_cols('L',  50,  66),
    'FR3' : aho_to_nt_cols('L',  67, 108),
    'CDR3': aho_to_nt_cols('L', 109, 138),
    'FR4' : aho_to_nt_cols('L', 139, 148),
}

H_CDR_V = H_REGIONS['CDR1'] + H_REGIONS['CDR2']
H_FWR_V = H_REGIONS['FR1']  + H_REGIONS['FR2'] + H_REGIONS['FR3']
H_ALL_V  = H_FWR_V + H_CDR_V

col_to_region_H = {}
for reg, cols in H_REGIONS.items():
    for c in cols:
        col_to_region_H[c] = reg

print(f"H V-gene: {len(H_ALL_V)//3} codons")

In [ ]:
GENETIC_CODE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
AA_LOOKUP = {0: 0}
for codon, aa in GENETIC_CODE.items():
    k = ord(codon[0]) * 65536 + ord(codon[1]) * 256 + ord(codon[2])
    AA_LOOKUP[k] = ord(aa)
_aa_vec = np.vectorize(lambda k: AA_LOOKUP.get(int(k), 0), otypes=[np.int32])

def translate_codon_array(nt):
    keys = nt[:, 0].astype(np.int32) * 65536 + nt[:, 1].astype(np.int32) * 256 + nt[:, 2].astype(np.int32)
    return _aa_vec(keys)

# Pre-compute expected R and S fractions for every codon string
# E[dN_frac]: fraction of single-nt changes from this codon that are replacement
# E[dS_frac]: fraction that are synonymous
# Used as the neutral expectation for dN/dS under uniform mutability
NT_LIST = list('ACGT')

def expected_rs_fractions(codon: str) -> tuple[float, float]:
    """Return (E_dN_frac, E_dS_frac) for a codon under uniform single-nt substitution."""
    ref_aa = GENETIC_CODE.get(codon, None)
    if ref_aa is None:
        return (np.nan, np.nan)
    n_R, n_S, n_total = 0, 0, 0
    for pos in range(3):
        for nt in NT_LIST:
            if nt == codon[pos]:
                continue
            mut = codon[:pos] + nt + codon[pos+1:]
            mut_aa = GENETIC_CODE.get(mut, None)
            if mut_aa is None:
                continue
            if mut_aa == '*' or ref_aa == '*':
                continue
            n_total += 1
            if mut_aa != ref_aa:
                n_R += 1
            else:
                n_S += 1
    if n_total == 0:
        return (np.nan, np.nan)
    return n_R / n_total, n_S / n_total

print("Genetic code and neutral expectations loaded.")

## S1 — Position-specific ω_i (dN/dS per Aho position)

For each Aho AA position i, across all memory sequences:
- `dN_obs(i)`: fraction of sequences with a replacement mutation at position i
- `dS_obs(i)`: fraction with a silent mutation at position i
- `E[dN(i)]`, `E[dS(i)]`: expected under neutral evolution (S5F mutability × codon structure)
- `ω_i = (dN_obs / E[dN]) / (dS_obs / E[dS])`

ω < 1 → purifying selection (structural constraint → Φ_S contributor)  
ω > 1 → positive selection (affinity-driven → Φ_A contributor)

In [ ]:
print("Loading alignment tables...")
mutated  = pl.read_parquet(MUTATED_FILE)
germline = pl.read_parquet(GERMLINE_FILE)

mutated_dedup  = mutated.unique(subset=[KEY], keep='first')
germline_dedup = germline.unique(subset=[KEY], keep='first')

# Memory sequences only
memory_names = set(memory[KEY].to_list())
mut_mem  = mutated_dedup.filter(pl.col(KEY).is_in(memory_names))
germ_mem = germline_dedup.filter(pl.col(KEY).is_in(memory_names))

# Build aligned joint for memory sequences
pos_cols = [c for c in mutated.columns if c != KEY]
aligned_mem = mut_mem.join(germ_mem, on=KEY, how='inner', suffix='_germ').sort(KEY)
print(f"Memory aligned joint: {aligned_mem.height:,}")

# H V-region arrays
H_V_COLS_SEQ  = [c for c in H_ALL_V if c in set(aligned_mem.columns) and f"{c}_germ" in set(aligned_mem.columns)]
H_V_COLS_GERM = [f"{c}_germ" for c in H_V_COLS_SEQ]

seq_HV  = aligned_mem.select(H_V_COLS_SEQ).fill_null(0).to_numpy().astype(np.int32)
germ_HV = aligned_mem.select(H_V_COLS_GERM).fill_null(0).to_numpy().astype(np.int32)
print(f"Arrays: {seq_HV.shape}")

In [ ]:
# Build per-position mutability weights from the S5F profile (Step 1 output)
# profile_H has one row per nt position; we need per-codon (AA position) S5F weight
# Use the SUM of 3-nt position mutabilities as the codon-level S5F weight
profile_dict = {
    row['nt_col']: row['mutability']
    for row in profile_H.to_dicts()
}

# Codon S5F weight: mean of the 3 nt positions
codon_s5f = []
for codon_i in range(len(H_V_COLS_SEQ) // 3):
    p0, p1, p2 = codon_i * 3, codon_i * 3 + 1, codon_i * 3 + 2
    m0 = profile_dict.get(H_V_COLS_SEQ[p0], 1.0)
    m1 = profile_dict.get(H_V_COLS_SEQ[p1], 1.0)
    m2 = profile_dict.get(H_V_COLS_SEQ[p2], 1.0)
    codon_s5f.append((m0 + m1 + m2) / 3.0)

codon_s5f = np.array(codon_s5f)
print(f"Codon S5F weights: min={codon_s5f.min():.3f}, mean={codon_s5f.mean():.3f}, max={codon_s5f.max():.3f}")

In [ ]:
# Per-codon dN/dS calculation
N = seq_HV.shape[0]
n_codons = len(H_V_COLS_SEQ) // 3

print(f"Computing ω for {n_codons} codons across {N:,} memory sequences...")
t0 = time.time()

records = []
for codon_i in range(n_codons):
    p0 = codon_i * 3
    nt_col = H_V_COLS_SEQ[p0]  # first nt of codon
    aho_aa = (int(nt_col[1:]) - 1) // 3 + 1
    region = col_to_region_H.get(nt_col, 'unknown')

    g = germ_HV[:, p0:p0+3]
    s = seq_HV[:,  p0:p0+3]

    # Valid: both germ and seq non-null, non-gap
    valid = (np.all(g > 0, axis=1) & np.all(g != GAP, axis=1) &
             np.all(s > 0, axis=1) & np.all(s != GAP, axis=1))
    n_valid = valid.sum()
    if n_valid < 10:
        continue

    has_mut = valid & np.any(g != s, axis=1)
    g_aa = translate_codon_array(g)
    s_aa = translate_codon_array(s)
    valid_aa = (g_aa > 0) & (s_aa > 0) & (g_aa != ord('*')) & (s_aa != ord('*'))

    n_R_obs = int((has_mut & (g_aa != s_aa) & valid_aa).sum())
    n_S_obs = int((has_mut & (g_aa == s_aa) & valid_aa).sum())

    # Consensus germline codon string for this position
    valid_idx = np.where(valid)[0]
    if len(valid_idx) == 0:
        continue
    g_sample = g[valid_idx[:1000]]  # use up to 1000 for consensus
    codon_counts = defaultdict(int)
    for row in g_sample:
        nt_str = ''.join(INT_TO_NT.get(int(x), '?') for x in row)
        if '?' not in nt_str:
            codon_counts[nt_str] += 1
    if not codon_counts:
        continue
    consensus_codon = max(codon_counts, key=codon_counts.get)

    # Expected R and S fractions under neutral evolution
    e_dn_frac, e_ds_frac = expected_rs_fractions(consensus_codon)
    if np.isnan(e_dn_frac) or e_dn_frac == 0 or e_ds_frac == 0:
        omega = np.nan
    else:
        # Scale expected counts by S5F mutability and number of sequences
        s5f_w = codon_s5f[codon_i]
        # Neutrally expected mutation rate ∝ s5f_w; split by codon structure
        e_dN = e_dn_frac * s5f_w
        e_dS = e_ds_frac * s5f_w

        # dN_obs and dS_obs as rates per sequence
        dN_obs = n_R_obs / n_valid
        dS_obs = n_S_obs / n_valid

        # ω = (dN_obs / E[dN]) / (dS_obs / E[dS]) = (dN_obs * E[dS]) / (dS_obs * E[dN])
        if dS_obs == 0:
            # Use pseudocount: add 0.5 synonymous mutations
            dS_obs = 0.5 / n_valid
        omega = (dN_obs * e_ds_frac) / (dS_obs * e_dn_frac)

    records.append({
        'nt_col':        nt_col,
        'aho_aa_pos':    aho_aa,
        'region':        region,
        'consensus_codon': consensus_codon,
        'n_valid':       int(n_valid),
        'n_R_obs':       n_R_obs,
        'n_S_obs':       n_S_obs,
        'dN_obs':        n_R_obs / n_valid,
        'dS_obs':        n_S_obs / n_valid,
        'e_dN_frac':     float(e_dn_frac) if not np.isnan(e_dn_frac) else None,
        'e_dS_frac':     float(e_ds_frac) if not np.isnan(e_ds_frac) else None,
        's5f_weight':    float(codon_s5f[codon_i]),
        'omega':         float(omega) if not np.isnan(omega) else None,
    })

omega_df = pl.DataFrame(records)
print(f"Done in {time.time()-t0:.1f}s — {omega_df.height} positions")
print(omega_df.select(['aho_aa_pos','region','n_R_obs','n_S_obs','omega']).describe())

In [ ]:
omega_df.write_parquet(OMEGA_FILE)
omega_df.write_csv(TABLES / "omega_per_position.csv")
print(f"Saved → {OMEGA_FILE}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Data ────────────────────────────────────────────────────────────────────
df = pl.read_csv(TABLES / "omega_per_position.csv")  # standalone-safe
df = df.filter(pl.col('omega').is_not_null())

plot_data = df.select(['aho_aa_pos', 'region', 'omega', 's5f_weight', 'n_valid'])
plot_data.write_csv(FIGURES / "fig_s1_omega_profile.csv")

x      = df['aho_aa_pos'].to_numpy()
y      = np.clip(df['omega'].to_numpy(), 0, 5)  # clip for display
regs   = df['region'].to_list()

# ── Figure ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

REGION_COLORS = {'CDR1':'#FFCCCC','CDR2':'#FFCCCC','FR1':'#E8F4FD','FR2':'#E8F4FD','FR3':'#E8F4FD'}
current_reg, reg_start = regs[0], x[0]
for i in range(1, len(regs)):
    if regs[i] != current_reg or i == len(regs) - 1:
        ax.axvspan(reg_start - 0.5, x[i-1] + 0.5, color=REGION_COLORS.get(current_reg,'#FFF'), alpha=0.35, lw=0)
        current_reg, reg_start = regs[i], x[i]

# Color by selection direction
colors = ['#E53935' if v > 1.0 else '#1E88E5' for v in y]
for xi, yi, ci in zip(x, y, colors):
    ax.vlines(xi, 0, yi, color=ci, alpha=0.6, lw=1.2)

ax.axhline(1.0, color='gray', linestyle='--', lw=1, label='ω=1 (neutral)')
ax.set_xlabel('Aho AA position')
ax.set_ylabel('ω (dN/dS, S5F-normalised)')
ax.set_title('Per-position ω across H V-region (memory sequences)')
ax.set_ylim(0, 5.5)

patches = [
    mpatches.Patch(color='#E53935', label='ω > 1 (positive selection)'),
    mpatches.Patch(color='#1E88E5', label='ω < 1 (purifying selection)'),
    mpatches.Patch(color='#FFCCCC', alpha=0.4, label='CDR'),
    mpatches.Patch(color='#E8F4FD', alpha=0.4, label='FWR'),
]
ax.legend(handles=patches, fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "fig_s1_omega_profile.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import matplotlib.pyplot as plt

# ── Data: ω distribution by region ──────────────────────────────────────────
df = pl.read_csv(TABLES / "omega_per_position.csv").filter(pl.col('omega').is_not_null())

omega_by_region = (
    df.group_by('region')
    .agg([
        pl.col('omega').mean().alias('mean_omega'),
        pl.col('omega').median().alias('median_omega'),
        pl.col('omega').std().alias('std_omega'),
        pl.len().alias('n_positions'),
    ])
    .sort('mean_omega')
)
omega_by_region.write_csv(FIGURES / "fig_s1_omega_by_region.csv")
print(omega_by_region)

# ── Figure ──────────────────────────────────────────────────────────────────
REGION_ORDER = ['FR1','CDR1','FR2','CDR2','FR3']
reg_dict = {r['region']: r for r in omega_by_region.to_dicts()}
orderedREG = [r for r in REGION_ORDER if r in reg_dict]

means  = [reg_dict[r]['mean_omega']   for r in orderedREG]
stds   = [reg_dict[r]['std_omega']    for r in orderedREG]
npos   = [reg_dict[r]['n_positions']  for r in orderedREG]
colors = ['#FFCCCC' if 'CDR' in r else '#E8F4FD' for r in orderedREG]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(orderedREG, means, yerr=stds, capsize=5,
              color=colors, edgecolor='#555', linewidth=0.6,
              error_kw={'elinewidth': 1.2})
for i, (r, n, m) in enumerate(zip(orderedREG, npos, means)):
    ax.text(i, m + stds[i] + 0.02, f'{n} pos', ha='center', va='bottom', fontsize=8)
ax.axhline(1.0, color='gray', linestyle='--', lw=1, label='ω=1 (neutral)')
ax.set_ylabel('Mean ω (± SD)')
ax.set_title('Mean dN/dS by V-region (memory sequences)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig_s1_omega_by_region.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## S2 — Forbidden Mutations

Positions where the S5F model predicts frequent mutations but observed rate is low:
`phi_S_filter[i] = log( expected_freq[i] / (observed_freq[i] + ε) )`

High Φ_S_filter = position generates mutations that selection eliminates → strong structural constraint.

In [ ]:
# For each codon position:
#   expected_freq = s5f_weight / sum(s5f_weights)  (share of mutations expected here)
#   observed_freq = (n_R_obs + n_S_obs) / (n_valid * sum(n_mut_per_seq))
#     Simplified: n_mut_at_pos / n_sequences_valid

epsilon = 1e-6

omega_data = pl.read_csv(TABLES / "omega_per_position.csv").filter(pl.col('omega').is_not_null())

total_s5f = omega_data['s5f_weight'].sum()

forbidden_records = []
for row in omega_data.to_dicts():
    exp_freq = row['s5f_weight'] / total_s5f
    obs_freq = (row['n_R_obs'] + row['n_S_obs']) / row['n_valid']
    phi_s_filter = np.log(exp_freq / (obs_freq + epsilon))
    forbidden_records.append({
        'nt_col':       row['nt_col'],
        'aho_aa_pos':   row['aho_aa_pos'],
        'region':       row['region'],
        'expected_freq': exp_freq,
        'observed_freq': obs_freq,
        'phi_S_filter': float(phi_s_filter),
        'omega':        row['omega'],
    })

forbidden_df = pl.DataFrame(forbidden_records).sort('phi_S_filter', descending=True)
print(f"Top 10 most constrained positions (highest Φ_S_filter):")
print(forbidden_df.head(10))

In [ ]:
forbidden_df.write_parquet(FORBID_FILE)
forbidden_df.write_csv(TABLES / "forbidden_mutations.csv")
print(f"Saved → {FORBID_FILE}")

In [ ]:
import matplotlib.pyplot as plt

# ── Data ────────────────────────────────────────────────────────────────────
df = pl.read_csv(TABLES / "forbidden_mutations.csv")  # standalone-safe

plot_data = df.select(['aho_aa_pos', 'region', 'expected_freq', 'observed_freq', 'phi_S_filter'])
plot_data.write_csv(FIGURES / "fig_s2_forbidden_scatter.csv")

# ── Figure ──────────────────────────────────────────────────────────────────
REGION_COLOR = {'CDR1':'#E53935','CDR2':'#E53935','FR1':'#1E88E5','FR2':'#1E88E5','FR3':'#1E88E5'}

x = df['expected_freq'].to_numpy()
y = df['observed_freq'].to_numpy()
regs = df['region'].to_list()
colors = [REGION_COLOR.get(r, '#888') for r in regs]

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(x, y, c=colors, s=15, alpha=0.7, linewidths=0)

# Diagonal = expected == observed
lim = max(x.max(), y.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=0.8, label='Expected = Observed')

import matplotlib.patches as mpatches
patches = [mpatches.Patch(color='#E53935', label='CDR'), mpatches.Patch(color='#1E88E5', label='FWR')]
ax.legend(handles=patches)
ax.set_xlabel('Expected mutation frequency (S5F)')
ax.set_ylabel('Observed mutation frequency (memory)')
ax.set_title('Forbidden mutations: expected vs observed per position\n(points below diagonal = structurally constrained)')
plt.tight_layout()
plt.savefig(FIGURES / "fig_s2_forbidden_scatter.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Data ────────────────────────────────────────────────────────────────────
df = pl.read_csv(TABLES / "forbidden_mutations.csv").sort('aho_aa_pos')  # standalone-safe

plot_data = df.select(['aho_aa_pos', 'region', 'phi_S_filter'])
plot_data.write_csv(FIGURES / "fig_s2_phi_s_profile.csv")

x    = df['aho_aa_pos'].to_numpy()
y    = df['phi_S_filter'].to_numpy()
regs = df['region'].to_list()

# ── Figure ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

REGION_COLORS = {'CDR1':'#FFCCCC','CDR2':'#FFCCCC','FR1':'#E8F4FD','FR2':'#E8F4FD','FR3':'#E8F4FD'}
current_reg, reg_start = regs[0], x[0]
for i in range(1, len(regs)):
    if regs[i] != current_reg or i == len(regs)-1:
        ax.axvspan(reg_start-0.5, x[i-1]+0.5, color=REGION_COLORS.get(current_reg,'#FFF'), alpha=0.35, lw=0)
        current_reg, reg_start = regs[i], x[i]

ax.bar(x, y, color=['#1E88E5' if v > 0 else '#E53935' for v in y], alpha=0.7, width=0.8, edgecolor='none')
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('Aho AA position')
ax.set_ylabel('Φ_S filter  [log(expected/observed)]')
ax.set_title('Structural constraint profile: Φ_S_filter per position\n(positive = constrained, negative = enriched beyond expectation)')

patches = [
    mpatches.Patch(color='#1E88E5', label='Constrained (Φ_S > 0)'),
    mpatches.Patch(color='#E53935', label='Enriched (Φ_S < 0)'),
    mpatches.Patch(color='#FFCCCC', alpha=0.4, label='CDR'),
    mpatches.Patch(color='#E8F4FD', alpha=0.4, label='FWR'),
]
ax.legend(handles=patches, fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "fig_s2_phi_s_profile.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## S3 — VH/VL Interface Co-variation (Mutual Information)

Vernier zone positions mediate VH/VL packing and are under coupled evolutionary constraint.

**VH Vernier positions (Aho AA):** 2, 47, 48, 67, 69, 71, 78, 93, 94  
**VL Vernier positions (Aho AA):** 2, 36, 46, 48, 49, 64, 66, 68, 71 (Chothia/Al-Lazikani)

For each pair (VH position i, VL position j): MI(i,j) = Σ p(a,b) log[p(a,b)/(p(a)·p(b))]

Miller-Madow bias correction: MI_corrected = MI_obs − (n_states − 1) / (2 · N)

In [ ]:
# Vernier zone Aho AA positions → AA columns in the master table
# The master table has iggnition-aligned nt columns; for AA-level co-variation
# we extract the amino acid from the sequence string (sequence_aa:0 / :1)
# OR we translate directly from the aligned nt codons.
# We use the nt arrays for consistency with the rest of the pipeline.

VERNIER_H_AA = [2, 47, 48, 67, 69, 71, 78, 93, 94]
VERNIER_L_AA = [2, 36, 46, 48, 49, 64, 66, 68, 71]

def aa_nt_cols(chain: str, aho_aa: int) -> list[str]:
    """Return the 3 nt column names for Aho AA position."""
    start = (aho_aa - 1) * 3 + 1
    return [f"{chain}{i}" for i in range(start, start + 3)]

V_H_NT = [aa_nt_cols('H', aa) for aa in VERNIER_H_AA]
V_L_NT = [aa_nt_cols('L', aa) for aa in VERNIER_L_AA]

# Flatten and filter to columns present in alignment
all_V_H = [c for triplet in V_H_NT for c in triplet]
all_V_L = [c for triplet in V_L_NT for c in triplet]

present = set(aligned_mem.columns)
all_V_H = [c for c in all_V_H if c in present]
all_V_L = [c for c in all_V_L if c in present]

print(f"VH Vernier nt cols: {len(all_V_H)} ({len(all_V_H)//3} positions)")
print(f"VL Vernier nt cols: {len(all_V_L)} ({len(all_V_L)//3} positions)")

In [ ]:
# Extract AA at each Vernier position for all memory sequences
# Translate from the nt arrays in aligned_mem

def extract_aa_col(aligned, nt_triplet_cols):
    """Extract AA ordinal for one codon position from aligned DataFrame."""
    available = [c for c in nt_triplet_cols if c in set(aligned.columns)]
    if len(available) < 3:
        return None
    arr = aligned.select(available[:3]).fill_null(0).to_numpy().astype(np.int32)
    # Zero rows where any nt is null or gap
    invalid = (arr[:, 0] == 0) | (arr[:, 1] == 0) | (arr[:, 2] == 0) | \
              (arr[:, 0] == GAP) | (arr[:, 1] == GAP) | (arr[:, 2] == GAP)
    aa = translate_codon_array(arr)
    aa[invalid] = 0
    return aa

# Collect VH and VL AA arrays per Vernier position
vh_aa_arrays = []
for aa_pos, triplet in zip(VERNIER_H_AA, V_H_NT):
    col = extract_aa_col(aligned_mem, triplet)
    if col is not None:
        vh_aa_arrays.append((aa_pos, col))

vl_aa_arrays = []
for aa_pos, triplet in zip(VERNIER_L_AA, V_L_NT):
    col = extract_aa_col(aligned_mem, triplet)
    if col is not None:
        vl_aa_arrays.append((aa_pos, col))

print(f"VH Vernier positions available: {len(vh_aa_arrays)}")
print(f"VL Vernier positions available: {len(vl_aa_arrays)}")

In [ ]:
def mutual_information_miller_madow(x: np.ndarray, y: np.ndarray) -> float:
    """Compute MI(X;Y) with Miller-Madow bias correction. Ignores zeros (invalid)."""
    # Keep only rows where both are valid (non-zero)
    valid = (x > 0) & (y > 0)
    x, y = x[valid], y[valid]
    N = len(x)
    if N < 50:
        return np.nan

    # Joint counts
    joint = defaultdict(int)
    for a, b in zip(x, y):
        joint[(int(a), int(b))] += 1

    # Marginals
    px = defaultdict(int)
    py = defaultdict(int)
    for (a, b), cnt in joint.items():
        px[a] += cnt
        py[b] += cnt

    MI = 0.0
    for (a, b), cnt in joint.items():
        p_ab = cnt / N
        p_a  = px[a] / N
        p_b  = py[b] / N
        if p_ab > 0:
            MI += p_ab * np.log(p_ab / (p_a * p_b))

    # Miller-Madow correction: subtract (n_nonzero_cells - 1) / (2N)
    n_cells = len(joint)
    MI_corrected = MI - (n_cells - 1) / (2 * N)
    return max(MI_corrected, 0.0)


print("Computing pairwise VH-VL Vernier MI...")
t0 = time.time()

mi_records = []
for h_pos, h_aa in vh_aa_arrays:
    for l_pos, l_aa in vl_aa_arrays:
        mi = mutual_information_miller_madow(h_aa, l_aa)
        mi_records.append({
            'VH_aa_pos': h_pos,
            'VL_aa_pos': l_pos,
            'MI_corrected': float(mi) if not np.isnan(mi) else None,
            'n_valid': int(((h_aa > 0) & (l_aa > 0)).sum()),
        })

mi_df = pl.DataFrame(mi_records)
print(f"Done in {time.time()-t0:.1f}s — {mi_df.height} pairs")
print(mi_df.sort('MI_corrected', descending=True).head(10))

In [ ]:
mi_df.write_csv(MI_FILE)
print(f"Saved → {MI_FILE}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Data ────────────────────────────────────────────────────────────────────
df = pl.read_csv(MI_FILE)  # standalone-safe

plot_data = df.select(['VH_aa_pos', 'VL_aa_pos', 'MI_corrected'])
plot_data.write_csv(FIGURES / "fig_s3_mi_heatmap.csv")

# Pivot to matrix
vh_positions = sorted(df['VH_aa_pos'].unique().to_list())
vl_positions = sorted(df['VL_aa_pos'].unique().to_list())

mi_matrix = np.zeros((len(vh_positions), len(vl_positions)))
df_dict = {(r['VH_aa_pos'], r['VL_aa_pos']): r['MI_corrected'] for r in df.to_dicts()}
for i, vh in enumerate(vh_positions):
    for j, vl in enumerate(vl_positions):
        val = df_dict.get((vh, vl))
        mi_matrix[i, j] = val if val is not None else 0.0

# ── Figure ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(mi_matrix, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='MI (bits, Miller-Madow corrected)')
ax.set_xticks(range(len(vl_positions)))
ax.set_xticklabels([f"VL{p}" for p in vl_positions], rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(vh_positions)))
ax.set_yticklabels([f"VH{p}" for p in vh_positions], fontsize=9)
ax.set_title('VH–VL Vernier zone mutual information\n(paired memory sequences)')
plt.tight_layout()
plt.savefig(FIGURES / "fig_s3_mi_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## Step 2 Summary

| Calculation | Output table | Output figure(s) |
|-------------|-------------|------------------|
| S1: ω per position | `omega_per_position.csv` | `fig_s1_omega_profile.png`, `fig_s1_omega_by_region.png` |
| S2: Forbidden mutations / Φ_S filter | `forbidden_mutations.csv` | `fig_s2_forbidden_scatter.png`, `fig_s2_phi_s_profile.png` |
| S3: VH/VL MI | `vh_vl_mutual_information.csv` | `fig_s3_mi_heatmap.png` |

**Interpretation for Φ_S:**
- `ω_i < 1` → purifying selection → `Φ_S(i) = -log(ω_i) > 0` (structural cost to mutate here)
- `phi_S_filter > 0` → generated mutations are eliminated → corroborates ω-based constraint
- High MI between VH/VL Vernier pairs → interface coupling → additional structural constraint

**Next step:** `03_phi_affinity.ipynb` — Affinity deficit Φ_A (A1–A3)